# Does excluding low-extent GFTM modes fix KBM mode selection?

Fusion_PhD-38e9.1. On the KBM Latin hypercubes most GFTM error is mode **selection**, not
identification: a mode close to GS2 is in GFTM's 4-mode spectrum but is not the fastest one.
Hypothesis: spurious fast modes with a short ballooning-angle **extent** win the argmax.

Data: the WIDTH-oracle scan (`wo_*` leaves; `docs/gyro_data_runs.md` in Fusion_PhD), GFTM at fixed
NBASIS_MAX=22 / NXGRID=28, FILTER=0.0, NMODES=4, 24 fixed widths 0.2-4.0, on SPR-045 (`KB_MODELS`)
and M1 (`KB_MODELS_MAST`). GS2 reference: the tail-averaged `pyro_cube_avg` bundles.
Extent: `pyrokinetics.diagnostics.extent.Extent` (branch `feature/extent_diagnostic`, 95% rule),
applied to each cube's `eigenfunctions`.

Sections: (a) the premise measured, (b) extent of spurious vs oracle modes, (c) the cutoff sweep,
(d) the tearing-parity selector as the baseline, plus three checks on the extent numbers.

**The oracle (mode closest to GS2) consults the answer: it is a bound, never an accuracy.**

## Imports and settings

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import pyrokinetics
from pyrokinetics.pyroscan import PyroScanGKOutput
from pyrokinetics.dataset_wrapper import DatasetWrapper
from pyrokinetics.diagnostics.extent import Extent

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "kbm_extent_filter"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
cases = {"SPR-045": "SPR-045", "M1": "M1"}  # database label -> case directory
scan_information = {"GFTM": "kbm_8d/wo_{width}", "GS2": "kbm_8d"}
output_dir = {"GFTM": "pyro_cube", "GS2": "pyro_cube_avg"}  # GS2: tail-averaged reference
output_file = "cube.nc"
widths = ["w0p2000", "w0p2297", "w0p2639", "w0p3031", "w0p3482", "w0p4000", "w0p4595",
          "w0p5279", "w0p6063", "w0p6965", "w0p8000", "w0p9000", "w1p0000", "w1p1000",
          "w1p2000", "w1p3000", "w1p4000", "w1p5000", "w1p6000", "w1p9218", "w2p3083",
          "w2p7726", "w3p3302", "w4p0000"]
width_value = {w: float(w[1:].replace("p", ".")) for w in widths}
expected_resolution = {"nbasis_used": 22, "nxgrid_used": 28}

fraction = 0.95                             # Extent's rule: theta range holding 95% of |Re field|^2
fields = ["phi", "apar"]                    # phi selects; apar reported alongside
cutoffs = np.round(np.arange(0, 12.01, 0.25), 2)       # absolute extent cutoff [rad]
rel_cutoffs = np.round(np.arange(0, 1.001, 0.05), 2)   # fraction of the case's largest unstable-mode extent
parity_threshold = 0.5   # GFTM FILTER=0.5: phi_norm_even < 0.5 flags tearing parity (gftm_LS.f90:582)
edge_amplitude = 0.05    # check 1: |phi| at the theta-grid edge above this fraction of its peak
example_width = "w1p0000"
n_examples = 3

## Load data

One `PyroScanGKOutput` per (database, width), read from the cube's netCDF. The extent branch has no
`PyroHypercube` or GFTM reader, so the cube is loaded as the scan's gk_output object, which carries
units. `Extent` adds `extent(sample, field, mode)` and `bounds(..., bound=lo/hi)` in place.

In [ ]:
gs2, gftm = {}, {}
for db, case in cases.items():
    gs2[db] = PyroScanGKOutput.from_netcdf(
        data_root / "GS2" / run_template / project / case / scan_information["GS2"] / output_dir["GS2"] / output_file)
    for w in widths:
        out = PyroScanGKOutput.from_netcdf(
            data_root / "GFTM" / run_template / project / case / scan_information["GFTM"].format(width=w)
            / output_dir["GFTM"] / output_file)
        Extent(out, fraction=fraction)
        gftm[db, w] = out
    print(db, "GS2 samples", gs2[db].data.sizes["sample"], "| GFTM cubes", len(widths))

## Calculate and select

Conventions, checked rather than assumed: both codes' growth rates are in the same pyrokinetics
units; each cube's audited resolution (`*_used`, from `out.gftm.localdump`) is the requested one.

Per case, GFTM returns 4 modes. A **selector** takes the argmax growth rate over `mode` among
*unstable* modes (gamma > 0) that pass its gate. A case where no unstable mode passes is a **MISS**:
it stays in the population with gamma_sel = 0, so its |log10| error is +inf and its absolute error is
gamma_GS2. Misses therefore move the median, and medlog becomes inf once they are the majority.

Population: cases whose GS2 growth rate is finite and positive (the same cases for every selector
and width, so every comparison is paired).

- `dominant`: no gate (c = 0; the current argmax over unstable modes).
- `oracle`: the unstable mode with the smallest |log10(gamma/gamma_GS2)| -- a bound.
- `extent >= c`: absolute cutoff on the 95% extent of phi (apar alongside).
- `extent >= r * max extent`: relative cutoff on the case's own longest unstable mode.
- `parity`: ballooning-parity gate, phi_norm_even >= 0.5, the post-hoc form of GFTM's FILTER=0.5
  (qp5.20). phi_norm_even is computed on the output theta grid as
  int |(phi(theta)+phi(-theta))/2|^2 / int |phi|^2; GFTM computes it from its Hermite coefficients,
  which is the same quantity for an even/odd basis but not identical after the output reconstruction.

In [ ]:
def mag(v):
    return np.asarray(getattr(v.data, "magnitude", v.data))


def select(gam, keep):
    # argmax growth over 'mode' among unstable modes passing `keep`; 0 where none passes (a MISS)
    g = np.where(keep & (gam > 0), gam, -np.inf)
    idx = g.argmax(axis=1)
    return np.where(np.isfinite(g.max(axis=1)), gam[np.arange(len(gam)), idx], 0.0), idx


def score(g_sel, g_ref):
    with np.errstate(divide="ignore"):
        log_err = np.abs(np.log10(g_sel / g_ref))
    return np.median(np.abs(g_sel - g_ref)), np.median(log_err), int((g_sel == 0).sum())


rows, sweep, cases_data = [], [], {}
for (db, w), out in gftm.items():
    d, ref = out.data, gs2[db].data
    assert str(d.growth_rate.data.units) == str(ref.growth_rate.data.units), "normalisations differ"
    for var, value in expected_resolution.items():
        assert (mag(d[var]) == value).all(), f"{db}/{w}: {var} is not {value}"
    assert np.allclose(mag(d.width_used), width_value[w])

    ref_by_name = dict(zip(ref.sample_name.values, mag(ref.growth_rate)))
    g_ref_all = np.array([ref_by_name.get(n, np.nan) for n in d.sample_name.values])
    ok = np.isfinite(g_ref_all) & (g_ref_all > 0)
    g_ref = g_ref_all[ok]
    gam = mag(d.growth_rate.transpose("sample", "mode"))[ok]
    ext = {f: mag(d.extent.sel(field=f).transpose("sample", "mode"))[ok] for f in fields}
    theta = d.theta.values
    phi = mag(d.eigenfunctions.sel(field="phi").transpose("sample", "theta", "mode"))[ok]

    unstable = gam > 0
    g_dom, i_dom = select(gam, True)
    with np.errstate(divide="ignore", invalid="ignore"):
        dist = np.where(unstable, np.abs(np.log10(gam / g_ref[:, None])), np.inf)
    i_orc = dist.argmin(axis=1)
    g_orc = np.where(unstable.any(axis=1), gam[np.arange(len(gam)), i_orc], 0.0)

    # the output theta grid is non-uniform and symmetric only to ~0.01 rad: reversal stands in for theta -> -theta
    assert np.abs(theta + theta[::-1]).max() < 0.1 * np.diff(theta).min(), "parity needs a symmetric theta grid"
    even = np.trapezoid(np.abs((phi + phi[:, ::-1]) / 2) ** 2, theta, axis=1) / np.trapezoid(np.abs(phi) ** 2, theta, axis=1)
    g_par, _ = select(gam, even >= parity_threshold)

    # check 1: 95% bounds within one grid spacing of the edge, or |phi| at the edge above edge_amplitude of peak
    lo, hi = (mag(d.bounds.sel(field="phi", bound=b).transpose("sample", "mode"))[ok] for b in ("lo", "hi"))
    dtheta = theta[1] - theta[0]
    at_edge = (lo <= theta[0] + dtheta) | (hi >= theta[-1] - dtheta)
    edge_amp = np.maximum(np.abs(phi[:, 0]), np.abs(phi[:, -1])) / np.abs(phi).max(axis=1)

    # check 2: rotate each eigenfunction so its peak is real, recompute Extent
    e = mag(d.eigenfunctions)
    axis = d.eigenfunctions.get_axis_num("theta")
    peak = np.take_along_axis(e, np.abs(e).argmax(axis=axis, keepdims=True), axis=axis)
    rot = DatasetWrapper(data_vars={"eigenfunctions": d.eigenfunctions.copy(data=e * np.exp(-1j * np.angle(peak)))})
    Extent(rot, fraction=fraction)
    with np.errstate(divide="ignore", invalid="ignore"):
        phase_change = {f: np.abs(mag(rot.data.extent.sel(field=f).transpose("sample", "mode"))[ok] - ext[f]) / ext[f]
                        for f in fields}
    del rot

    differ = i_dom != i_orc
    with np.errstate(divide="ignore"):
        err_dom, err_orc = np.abs(np.log10(g_dom / g_ref)), np.abs(np.log10(g_orc / g_ref))
    r = np.arange(len(gam))
    row = dict(db=db, width=width_value[w], n=int(ok.sum()), n_gs2_stable_or_nan=int((~ok).sum()),
               differ=int(differ.sum()),
               medlog_dom=score(g_dom, g_ref)[1], medlog_orc=score(g_orc, g_ref)[1],
               medlog_par=score(g_par, g_ref)[1], miss_par=score(g_par, g_ref)[2],
               medAE_dom=score(g_dom, g_ref)[0], medAE_orc=score(g_orc, g_ref)[0], medAE_par=score(g_par, g_ref)[0],
               medlog_dom_agree=np.median(err_dom[~differ]), medlog_dom_differ=np.median(err_dom[differ]),
               medlog_orc_differ=np.median(err_orc[differ]),
               share_err_from_differ=err_dom[differ & np.isfinite(err_dom)].sum() / err_dom[np.isfinite(err_dom)].sum(),
               unstable_modes=int(unstable.sum()), unstable_nan_extent=int((unstable & ~np.isfinite(ext["phi"])).sum()),
               frac_bounds_at_edge=at_edge[unstable].mean(), frac_edge_amp=(edge_amp[unstable] > edge_amplitude).mean(),
               **{f"phase_{f}_median_rel_change": np.nanmedian(phase_change[f][unstable]) for f in fields},
               **{f"phase_{f}_frac_over_10pct": (phase_change[f][unstable] > 0.1).mean() for f in fields})
    for f in fields:
        a, b = ext[f][r, i_dom][differ], ext[f][r, i_orc][differ]
        row[f"{f}_ext_dom_shorter"] = (a < b).mean()
        row[f"{f}_ext_ratio_median"] = np.median(a / b)
    rows.append(row)

    longest = np.nanmax(np.where(unstable, ext["phi"], np.nan), axis=1, initial=0)
    for f in fields:
        for kind, grid in (("abs", cutoffs), ("rel", rel_cutoffs)):
            for c in grid:
                threshold = c if kind == "abs" else c * longest[:, None]
                keep = (ext[f] >= threshold) | (c == 0)   # c = 0 keeps NaN-extent modes: exactly `dominant`
                medAE, medlog, miss = score(select(gam, keep)[0], g_ref)
                sweep.append(dict(db=db, width=width_value[w], field=f, kind=kind, cutoff=c,
                                  medlog=medlog, medAE=medAE, miss=miss, n=len(g_ref)))
    cases_data[db, w] = dict(names=d.sample_name.values[ok], gam=gam, g_ref=g_ref, i_dom=i_dom, i_orc=i_orc,
                             ext=ext, err_dom=err_dom, differ=differ)

summary = pd.DataFrame(rows)
sweep = pd.DataFrame(sweep)

### (a) The premise, measured

`differ`: cases where the oracle mode is not the dominant one. `share_err_from_differ`: fraction of
the dominant arm's total |log10| error carried by those cases.

In [ ]:
pd.set_option("display.width", 250, "display.max_columns", 40)
print(summary[["db", "width", "n", "n_gs2_stable_or_nan", "differ", "medlog_dom", "medlog_orc",
               "medlog_dom_agree", "medlog_dom_differ", "medlog_orc_differ", "share_err_from_differ",
               "medAE_dom", "medAE_orc"]].round(4).to_string(index=False))
print(summary.groupby("db")[["differ", "share_err_from_differ"]].agg(["min", "median", "max"]).round(3))

### Checks on the extent numbers (grid edge, phase, NaN extents)

In [ ]:
print(summary[["db", "width", "unstable_modes", "unstable_nan_extent", "frac_bounds_at_edge", "frac_edge_amp",
               ] + [f"phase_{f}_{k}" for f in fields for k in ("median_rel_change", "frac_over_10pct")]].round(4).to_string(index=False))

### (b) Extent of the spurious dominant mode vs the oracle mode, where they differ

In [ ]:
print(summary[["db", "width", "differ"] + [f"{f}_{k}" for f in fields for k in ("ext_dom_shorter", "ext_ratio_median")]]
      .round(3).to_string(index=False))
for db in cases:
    for f in fields:
        a = np.concatenate([v["ext"][f][np.arange(len(v["gam"])), v["i_dom"]][v["differ"]] for (d_, w), v in cases_data.items() if d_ == db])
        b = np.concatenate([v["ext"][f][np.arange(len(v["gam"])), v["i_orc"]][v["differ"]] for (d_, w), v in cases_data.items() if d_ == db])
        print(f"{db} {f}: pooled over widths n={len(a)}, dominant shorter in {np.mean(a < b):.3f}, "
              f"median ratio dom/orc {np.nanmedian(a / b):.3f}")

### (c) + (d) The cutoff sweep against the parity baseline

For each (database, width): the cutoff minimising medlog, its miss count and change vs c = 0,
next to the parity selector and the oracle bound.

In [ ]:
best = (sweep[np.isfinite(sweep.medlog)].sort_values("medlog").groupby(["db", "field", "kind", "width"]).head(1)
        .set_index(["db", "field", "kind", "width"]).sort_index())
base = sweep[sweep.cutoff == 0].set_index(["db", "field", "kind", "width"])
best = best.assign(medlog_c0=base.medlog, medAE_c0=base.medAE)
best = best.assign(d_medlog=best.medlog - best.medlog_c0)
best = best.join(summary.set_index(["db", "width"])[["medlog_par", "miss_par", "medlog_orc"]])
for f in fields:
  print(f"--- {f}, absolute cutoff ---")
  print(best.loc[(slice(None), f, "abs"), ["cutoff", "medlog", "medlog_c0", "d_medlog", "medAE", "medAE_c0",
                                             "miss", "medlog_par", "miss_par", "medlog_orc"]].round(4).to_string())
print("--- phi and apar, relative cutoff ---")
print(best.loc[(slice(None), slice(None), "rel"), ["cutoff", "medlog", "medlog_c0", "d_medlog", "miss"]].round(4).to_string())
for f in fields:
    for kind in ("abs", "rel"):
        b = best.loc[(slice(None), f, kind)]
        wins = b[b.d_medlog < 0]
        print(f"{f}/{kind}: cells where some cutoff lowers medlog: "
              + ", ".join(f"{db} {len(wins.loc[db]) if db in wins.index.get_level_values(0) else 0}/{len(widths)}" for db in cases)
              + f" | largest improvement: " + ", ".join(f"{db} {b.loc[db].d_medlog.min():+.4f}" for db in cases))

## Plot

In [ ]:
cmap = plt.get_cmap("viridis")
colour = {w: cmap(i / (len(widths) - 1)) for i, w in enumerate(widths)}

fig1, axes1 = plt.subplots(len(fields), 2, figsize=(12, 9), sharey=True)
for ax, (f, db) in zip(axes1.flat, [(f, db) for f in fields for db in cases]):
    for w in widths:
        s = sweep[(sweep.db == db) & (sweep.width == width_value[w]) & (sweep.field == f) & (sweep.kind == "abs")]
        ax.plot(s.cutoff, s.medlog, color=colour[w], marker="", lw=1.2)
        ax.axhline(summary[(summary.db == db) & (summary.width == width_value[w])].medlog_orc.item(),
                   color=colour[w], ls="--", lw=0.8)
    ax.set(xlabel=f"extent cutoff $c$ on {f} [rad]", title=f"{db}, {f} extent (n={s.n.iloc[0]}; dashed: oracle bound)")
for ax in axes1[:, 0]:
    ax.set(ylabel=r"median $|\log_{10}(\gamma_{\rm GFTM}/\gamma_{\rm GS2})|$", ylim=(0, None))
fig1.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, len(widths) - 1), cmap=cmap), ax=axes1,
              ticks=range(0, len(widths), 3), label="WIDTH").ax.set_yticklabels([widths[i][1:].replace("p", ".") for i in range(0, len(widths), 3)])
plt.show()

In [ ]:
fig2, axes2 = plt.subplots(2 * len(fields), 2, figsize=(13, 17), width_ratios=[12, 1], sharey="row")
for row, (f, db) in enumerate([(f, db) for f in fields for db in cases]):
    s = sweep[(sweep.db == db) & (sweep.field == f) & (sweep.kind == "abs")]
    grid = s.pivot(index="width", columns="cutoff", values="medlog")
    delta = grid.sub(grid[0.0], axis=0)
    miss = s.pivot(index="width", columns="cutoff", values="miss") / s.n.iloc[0]
    par = summary[summary.db == db].set_index("width")
    lim = np.nanmax(np.abs(np.where(np.isfinite(delta), delta, np.nan)))
    norm = TwoSlopeNorm(0, -lim, lim)
    y = np.arange(len(widths))
    m = axes2[row, 0].pcolormesh(cutoffs, y, np.ma.masked_invalid(delta.values), cmap="RdBu_r", norm=norm, shading="nearest")
    cs = axes2[row, 0].contour(cutoffs, y, miss.values, levels=[0.05, 0.25, 0.5], colors="k", linewidths=0.8)
    axes2[row, 0].clabel(cs, fmt=lambda v: f"{v:.0%} miss", fontsize=8)
    axes2[row, 0].set(xlabel=f"extent cutoff $c$ on {f} [rad]", ylabel="WIDTH", yticks=y[::2],
                      yticklabels=[f"{v:g}" for v in grid.index[::2]],
                      title=f"{db}, {f} extent: medlog change vs c=0 (blue = better; grey = misses dominate)")
    axes2[row, 0].set_facecolor("0.8")
    axes2[row, 1].pcolormesh([0], y, (par.medlog_par - par.medlog_dom).values[:, None], cmap="RdBu_r", norm=norm, shading="nearest")
    axes2[row, 1].set(xticks=[0], xticklabels=["parity\nselector"])
    axes2[row, 1].grid(False)
    axes2[row, 0].grid(False)
    fig2.colorbar(m, ax=axes2[row, :], label=r"$\Delta$ medlog")
plt.show()

In [ ]:
fig3, axes3 = plt.subplots(2, 2, figsize=(12, 9.5))
for col, db in enumerate(cases):
    pts = {f: [] for f in fields}
    for w in widths:
        v = cases_data[db, w]
        r = np.arange(len(v["gam"]))
        for f in fields:
            pts[f].append((v["ext"][f][r, v["i_dom"]][v["differ"]], v["ext"][f][r, v["i_orc"]][v["differ"]],
                           np.full(v["differ"].sum(), width_value[w])))
    a, b, wv = (np.concatenate(x) for x in zip(*pts["phi"]))
    sc = axes3[0, col].scatter(b, a, c=np.log(wv), cmap=cmap, s=6, alpha=0.6)
    axes3[0, col].plot([0.3, 60], [0.3, 60], color="0.3", lw=1, marker="")
    axes3[0, col].set(xscale="log", yscale="log", xlabel=r"oracle-mode $\phi$ extent [rad]",
                      ylabel=r"dominant (spurious) $\phi$ extent [rad]",
                      title=f"{db}: {len(a)} (case, width) pairs where they differ, "
                            f"dominant shorter {np.mean(a < b):.0%}")
    for f, ls in zip(fields, ("-", "--")):
        a_, b_, _ = (np.concatenate(x) for x in zip(*pts[f]))
        axes3[1, col].hist(np.log2(a_ / b_), bins=60, range=(-5, 5), histtype="step", ls=ls, lw=1.5,
                           label=f"{f}: median {np.nanmedian(np.log2(a_ / b_)):+.2f}")
    axes3[1, col].axvline(0, color="0.3", lw=1)
    axes3[1, col].set(xlabel=r"$\log_2$(dominant extent / oracle extent)", ylabel="(case, width) pairs")
    axes3[1, col].legend()
fig3.colorbar(sc, ax=axes3[0, :], label="ln WIDTH")
plt.show()

In [ ]:
fig4, axes4 = plt.subplots(len(cases), n_examples, figsize=(15, 8), sharex=True)
for row, db in enumerate(cases):
    v = cases_data[db, example_width]
    d = gftm[db, example_width].data
    ref = gs2[db].data
    picks = np.where(v["differ"] & np.isfinite(v["err_dom"]))[0]
    picks = picks[np.argsort(v["err_dom"][picks])[::-1][:n_examples]]   # largest dominant-mode errors
    for ax, k in zip(axes4[row], picks):
        name = v["names"][k]
        s = int(np.where(d.sample_name.values == name)[0][0])
        for label, m, c in (("dominant", v["i_dom"][k], "C3"), ("oracle", v["i_orc"][k], "C0")):
            e = np.abs(mag(d.eigenfunctions.sel(field="phi").isel(sample=s, mode=m)))
            lo, hi = mag(d.bounds.sel(field="phi").isel(sample=s, mode=m))
            ax.plot(d.theta, e / e.max(), color=c, marker="", label=f"{label}: $\\gamma$={v['gam'][k, m]:.3f}, ext={hi - lo:.2f}")
            ax.axvspan(lo, hi, color=c, alpha=0.12)
        sg = int(np.where(ref.sample_name.values == name)[0][0])
        e = np.abs(mag(ref.eigenfunctions.sel(field="phi").isel(sample=sg)))
        ax.plot(ref.theta, e / e.max(), color="black", marker="", label=f"GS2: $\\gamma$={v['g_ref'][k]:.3f}")
        ax.set(title=f"{db} {name}, W={width_value[example_width]:g}", xlabel=r"$\theta$ [rad]", xlim=(-15, 15))
        ax.legend(fontsize=8)
    axes4[row, 0].set(ylabel=r"$|\phi(\theta)|$ / peak")
plt.show()

## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig1.savefig(output_dir / "fig1_medlog_vs_cutoff.png")
fig2.savefig(output_dir / "fig2_heatmap_width_cutoff.png")
fig3.savefig(output_dir / "fig3_extent_spurious_vs_oracle.png")
fig4.savefig(output_dir / "fig4_example_eigenfunctions.png")

## Interpretation

See the Fusion_PhD `results/kbm_extent_filter/README.md` for the verdict written from this notebook's
executed output. Limitations: the oracle consults GS2 and is a bound; extents come from pyro's fixed
output theta grid (+-9 pi), so edge-limited modes (check 1) give lower bounds; |Re(phi)| makes extent
phase-dependent (check 2); the parity baseline is computed on the output grid rather than from
GFTM's Hermite coefficients.